# 🧠 GIADA — CC-01–CC-04 · Contratto, costo e memoria
N1 del dropshot CC: nessun training. Contratto esplicito; baseline Task35; otto nuove ancore native con sinapsi; 288 continuazioni NEURON da storie di clamp controllate; benchmark GPU del core residente.
Emendamento prospettico: le collisioni usano ora direttamente i futuri NEURON. Il floor della mappa analitica sugli stati perturbati viene riportato separatamente, conservando i due esiti precedenti e tutte le soglie.
Le collisioni V/Ca e delle finestre recenti sono confronti causali a ingressi identici. I target futuri servono alla misura, mai come input del modello. I risultati Task35 già aperti sono baseline storiche. Il costo del core GPU e quello del generatore sinaptico CPU vengono riportati con perimetri espliciti.
**Input Kaggle: nessuno. GPU T4 e Internet richiesti.** Codice, protocollo e notebook devono provenire dalla stessa revisione. I processi nativi e CUDA sono isolati. Output sintetico per evitare flooding.


In [ ]:
import importlib.metadata, json, os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/kaggle/working/giada_cc_n1'); WORK.mkdir(parents=True,exist_ok=True)
REPO,TEACHER=WORK/'giada',WORK/'neuron_as_deep_net'
BRANCH='codex/surrogate-validity-audit'
if not REPO.exists(): subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',BRANCH],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
EXPECTED_REVISION=os.environ.get('GIADA_CC_N1_REVISION')
if EXPECTED_REVISION: subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXPECTED_REVISION],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
CANONICAL=json.loads((REPO/'notebooks/cc_n1_contract_memory.ipynb').read_text())
assert CANONICAL['metadata']['giada_contract']=='giada-cc-n1-contract-memory-v1'
if not TEACHER.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
TEACHER_REV='074c4666300a8ad246601dab179a97a6942f0f29'
if subprocess.check_output(['git','-C',str(TEACHER),'rev-parse','HEAD'],text=True).strip()!=TEACHER_REV: subprocess.run(['git','-C',str(TEACHER),'fetch','--depth','1','origin',TEACHER_REV],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach',TEACHER_REV],check=True)
try: installed=importlib.metadata.version('NEURON').split('+')[0]
except importlib.metadata.PackageNotFoundError: installed=None
if installed!='8.2.7': subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
import torch
assert torch.cuda.is_available(),'GPU CUDA richiesta per la mappa dei costi.'
print({'revision':REVISION,'teacher':TEACHER_REV,'gpu':torch.cuda.get_device_name(),'inputs_required':0})


## 📋 CC-01 — Contratto congelato prima del run
Le soglie del nuovo candidato sono definite nel protocollo versionato: V pooled ≤0,05 mV, episodio ≤0,1 mV, Ca ≤5e-6 mM, più margini rispetto alla baseline appaiata. Questo notebook misura riferimenti e insufficienze informative; la conferma comportamentale del candidato arriverà dopo. Stochasticità, inizializzazione e passo sono espliciti.


In [ ]:
SPEC=json.loads((REPO/'experiments/cc_n1_contract_memory_preregistration_v1.json').read_text())
assert SPEC['roadmap_ids']==['CC-01','CC-02','CC-03','CC-04']
print({'tasks':SPEC['roadmap_ids'],'dt_ms':SPEC['dt_ms'],'future_candidate_limits':SPEC['cc01_contract']['future_candidate_limits'],'training':False})


## 🧪 CC-02–CC-04 — Esecuzione isolata
Prima i controlli nativi, poi CUDA. La preparazione a clamp genera stati fisici controllati, senza assumere che siano frequenti nella fisiologia. Per una finestra recente interamente contenuta nella coda comune, V e Ca sono identici ma la memoria dei gate può ancora differire. Il report salva separazioni dei futuri, controlli di trapianto dello stato e curve per durata della coda.


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts/giada_cc_n1_contract_memory')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}; usa una sessione pulita.'
LOG=Path('/kaggle/working/cc_n1_process.log')
command=[sys.executable,'-u',str(REPO/'scripts/run_cc_n1_contract_memory.py'),'--teacher',str(TEACHER),'--output',str(OUTPUT)]
with subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process, LOG.open('w') as log:
    for line in process.stdout:
        log.write(line); log.flush()
        if line.startswith('[GIADA CC-N1]'): print(line.rstrip(),flush=True)
    RETURN_CODE=process.wait()
OUTPUT.mkdir(parents=True,exist_ok=True)
shutil.copy2(LOG,OUTPUT/'process.log')
report_path=next((OUTPUT/name for name in ('final_report.json','failure_report.json') if (OUTPUT/name).is_file()),None)
REPORT=json.loads(report_path.read_text()) if report_path else {'valid':False,'diagnosis':'PROCESS_TERMINATED_WITHOUT_REPORT','process_exit':RETURN_CODE}
if report_path is None: (OUTPUT/'failure_report.json').write_text(json.dumps(REPORT,indent=2))
print({k:REPORT.get(k) for k in ('valid','diagnosis','error','n2_preparation_recommended','training_performed')})
if RETURN_CODE: print('Ultime righe diagnostiche:\n'+'\n'.join(LOG.read_text(errors='replace').splitlines()[-18:]))
ARCHIVE=shutil.make_archive('/kaggle/working/giada_cc_n1_contract_memory','zip',root_dir=OUTPUT)
print({'archive':ARCHIVE,'process_exit':RETURN_CODE})


In [ ]:
from IPython.display import display
if (OUTPUT/'memory_analysis.json').is_file():
    memory=json.loads((OUTPUT/'memory_analysis.json').read_text())
    display({'instantaneous_collision':memory['instantaneous_collision_detected'],'history_windows':memory['history_windows'],'controls':memory['controls'],'minimum_latent_dimension_identified':False})
if (OUTPUT/'gpu_report.json').is_file():
    import pandas as pd
    gpu=json.loads((OUTPUT/'gpu_report.json').read_text())
    display(pd.DataFrame([{k:r[k] for k in ('arm','batch_per_seed','mode','parameter_count_per_seed','median_wall_ms_per_step_ensemble','ns_per_compartment_step')} for r in gpu['rows']]))
    print({'cost_map_valid':gpu['valid'],'compile_failures':len(gpu['compile_failures']),'speedup_claim':False})


## 💾 Download facoltativo dello ZIP
Il salvataggio Kaggle conserva già l'archivio. Per scaricarlo dal browser, eseguire la cella seguente: il metodo è quello usato nel progetto (base64 → Blob → download). Nei run autonomi questa cella resta inattiva.


In [ ]:
DOWNLOAD_IN_BROWSER=False
if DOWNLOAD_IN_BROWSER:
    import base64
    from IPython.display import Javascript
    encoded=base64.b64encode(Path(ARCHIVE).read_bytes()).decode('ascii')
    display(Javascript('const raw=atob('+json.dumps(encoded)+'); const bytes=Uint8Array.from(raw,c=>c.charCodeAt(0)); const url=URL.createObjectURL(new Blob([bytes],{type:"application/zip"})); const a=document.createElement("a"); a.href=url; a.download="giada_cc_n1_contract_memory.zip"; document.body.appendChild(a); a.click(); a.remove(); setTimeout(()=>URL.revokeObjectURL(url),30000);'))
